In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window

def transform_Sales_territory(Sales_territory_df):

    windowSpec_rw = Window.partitionBy("TerritoryID").orderBy("ModifiedDate")
    Sales_territory_df = Sales_territory_df.withColumn("Group",F.when(F.col("Group").isNull(), F.lit("Unknown")).otherwise(F.col("Group"))).withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    Sales_territory_df = Sales_territory_df.filter(F.col("rw") == 1).drop("rw")
    Sales_territory_df = Sales_territory_df.withColumn("processed_timestamp", F.current_timestamp())
    Sales_territory_df = Sales_territory_df.select(
      F.col("TerritoryID").cast(IntegerType()).alias("TerritoryID"),
      F.trim(F.col("Name")).cast(StringType()).alias("Name"),
      F.trim(F.col("CountryRegionCode")).cast(StringType()) .alias("CountryRegionCode"),
      F.col("Group").cast(StringType()).alias("Group"),
      F.col("SalesYTD").cast(DecimalType(19,4)).alias("SalesYTD"),
      F.col("SalesLastYear").cast(DecimalType(19,4)).alias("SalesLastYear"),
      F.col("CostYTD").cast(DecimalType(19,4)).alias("CostYTD"),
      F.col("CostLastYear").cast(DecimalType(19,4)).alias("CostLastYear"),
      F.col("rowguid").cast(StringType()).alias("rowguid"),
      F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
      F.col("processed_timestamp")
    )
                                 
    return Sales_territory_df



if __name__ == "__main__":

    Sales_territory_tbl = dbutils.widgets.get("sales_territory")
    Sales_territory_df = df = spark.read.table(Sales_territory_tbl)
    Sales_territory_df_tgt = transform_Sales_territory(Sales_territory_df)
    sales_territory_slv_tbl = dbutils.widgets.get("sales_territory_tgt")
    Sales_territory_df_tgt.write.mode("overwrite").format("delta").partitionBy("ModifiedDate").saveAsTable(sales_territory_slv_tbl)